# J3: graph model on the "humanise a cycle" task

Trains the graph model (variants V0 = no symmetry, V1 = beat symmetry C4, V3 = full rotation C12) on leave-one-take-out folds and compares it with the reference models B1 and B2 on the same strokes. Metrics and settings go to your Weights & Biases project; no corpus data leaves this session.

**Before running:** in Colab's left bar, open **Secrets** (key icon), add `WANDB_API_KEY` with your key from https://wandb.ai/authorize, and turn on *Notebook access*. Then *Runtime > Change runtime type > GPU* (optional, the model is small) and *Run all*.


In [ ]:
# Settings
REPO = "https://github.com/clementdumeril/Djembe.git"
REF = "main"                      # pin a commit hash here to reproduce a given run exactly
WANDB_PROJECT = "djembe"          # set to None to run without Weights & Biases
VARIANTS = "V0 V1 V3"
SEEDS = "0 1 2"
EPOCHS = 40
USE_DRIVE_CACHE = True     # keep a private copy of the annotations in your Drive (OSF is sometimes unavailable)


In [ ]:
import os, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine, slower)")
if not os.path.exists("/content/Djembe"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/Djembe"], check=True)
subprocess.run(["git", "-C", "/content/Djembe", "checkout", "-q", REF], check=True)
%cd /content/Djembe
!pip -q install -e . wandb pytest
!git log --oneline -1


In [ ]:
# Annotations only (about 10 MB, CC BY-NC-ND 4.0): a private copy for your own use, never redistributed.
import shutil
CACHE = "/content/drive/MyDrive/djembe_corpus"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(CACHE):
        shutil.copytree(CACHE, "/content/corpus", dirs_exist_ok=True)
        print("annotations copied from your Drive")
!python -m djembe.dataset download --corpus /content/corpus
assert _exit_code == 0, "Download from OSF failed even after retries: run this cell again in a few minutes."
if USE_DRIVE_CACHE:
    shutil.copytree("/content/corpus", CACHE, dirs_exist_ok=True)
!python -m djembe.dataset build --corpus /content/corpus
assert _exit_code == 0, "Building the event table failed: see the message above."
!DJEMBE_CORPUS=/content/corpus python -m pytest -q tests
assert _exit_code == 0, "Tests failed: see the message above."


In [ ]:
if WANDB_PROJECT:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # read from Colab Secrets, never printed
    import wandb
    wandb.login()


In [ ]:
flag = f"--wandb-project {WANDB_PROJECT}" if WANDB_PROJECT else ""
!python experiments/gnn/train_humanize.py --variants {VARIANTS} --seeds {SEEDS} --epochs {EPOCHS} {flag} --out results/j3_humanize.json
assert _exit_code == 0, "Training stopped with an error: scroll up in this cell to the first 'Traceback' and send it."


In [ ]:
import json, os
assert os.path.exists("results/j3_humanize.json"), "No results yet: the training cell above did not finish."
s = json.load(open("results/j3_humanize.json"))["summary"]
for name, v in s.items():
    print(f"{name:16s} offsets {v['nll_delta']:+.3f}   intensity {v['nll_intensity']:.3f}   ({v['runs']} runs)")
# To keep the metrics: download results/j3_humanize.json from the file panel, or copy it to your Drive.
